# A/B Testing – The ShopNow Checkout Experiment 🛒🎲

This notebook complements the lecture slides in `slides/ab_testing/index.html`.

## Story
**ShopNow** built a new one-page checkout (variant **B**). The old three-page checkout is variant **A**.
Today **5.0 %** of checkout visitors buy. The product team believes B lifts this by **at least 0.5 percentage points** (to 5.5 %).

We are the data team. Our job:

1. **Plan** the experiment: how many users do we need?
2. **Simulate** the experiment (we play the role of the website and the customers).
3. **Analyze** it: is the difference real, or just noise?
4. See what goes wrong when you **peek** every day.
5. Analyze a **continuous metric** (revenue per visitor).
6. Run the **sanity checks** every experiment needs.

Why simulate instead of using a real dataset? Because in a simulation **we know the true effect**, so we can check whether the test finds it. That is the best way to build intuition for noise.

## Tools
- `statsmodels` for sample size and the test for proportions
- `scipy.stats` for the t-test and the chi-squared test
- `pandas` and `matplotlib` as usual

In [ ]:
# Colab has these preinstalled. If you run locally: pip install statsmodels scipy pandas matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import (
    proportion_effectsize,
    proportions_ztest,
    confint_proportions_2indep,
)
from scipy import stats

pd.set_option("display.precision", 4)

## Exercise 1: Plan the sample size 📏

Four inputs, three of them are decisions:

| knob | ShopNow | who decides |
|---|---|---|
| baseline conversion rate | 5.0 % | the data (BI dashboard) |
| minimum detectable effect (MDE) | +0.5 pp (5.0 % → 5.5 %) | business |
| significance level α | 5 % | convention |
| power (1 − β) | 80 % | convention |

In [ ]:
BASELINE = 0.05   # 5.0 % of checkout visitors buy today
MDE = 0.005       # we want to detect at least +0.5 percentage points
ALPHA = 0.05
POWER = 0.80
VISITORS_PER_DAY = 2000   # checkout visitors per day, both groups together

In [ ]:
# Task 1: Compute the required number of users PER GROUP.
effect = proportion_effectsize(BASELINE + MDE, BASELINE)
n_per_group = NormalIndPower().solve_power(
    effect_size=effect, alpha=ALPHA, power=POWER, alternative="two-sided"
)
print(f"effect size (Cohen's h): {effect:.4f}")
print(f"users per group: {n_per_group:,.0f}")

In [ ]:
# Task 2: How many days does the experiment take with VISITORS_PER_DAY visitors (split 50/50)?
days = 2 * n_per_group / VISITORS_PER_DAY
print(f"total users: {2 * n_per_group:,.0f}  ->  about {days:.0f} days")

In [ ]:
# Task 3: Plot the power curve: power (y) vs users per group (x) for an MDE of +0.5 pp and of +1.0 pp.
ns = np.arange(1_000, 60_000, 1_000)

plt.figure(figsize=(8, 4))
for mde in [0.005, 0.01]:
    es = proportion_effectsize(BASELINE + mde, BASELINE)
    power = [NormalIndPower().power(effect_size=es, nobs1=n, alpha=ALPHA, ratio=1.0) for n in ns]
    plt.plot(ns, power, label=f"MDE +{mde*100:.1f} pp")
plt.axhline(POWER, ls="--", color="grey", label="80 % power")
plt.xlabel("users per group")
plt.ylabel("power")
plt.legend()
plt.show()

In [ ]:
# Task 4: The business says: "We can only run this for 14 days."
n_14_days = 14 * VISITORS_PER_DAY / 2   # per group

for mde in np.arange(0.001, 0.0201, 0.001):
    es = proportion_effectsize(BASELINE + mde, BASELINE)
    pw = NormalIndPower().power(effect_size=es, nobs1=n_14_days, alpha=ALPHA, ratio=1.0)
    if pw >= POWER:
        print(f"with {n_14_days:,.0f} users per group we can detect +{mde*100:.1f} pp (power {pw:.0%})")
        break

# Interpretation: in two weeks we can only detect a lift of about +1 pp (20 % relative).
# If the business only expects +0.5 pp, a 14-day test is pointless: it would miss the effect most of the time.

## Exercise 2: Simulate the experiment 🎲

We play the website: every visitor gets a coin flip (A or B), then "decides" to buy with the probability of their group.

The function below is almost complete. It also simulates the device of the user and the basket value of buyers, which we need later.

In [ ]:
def simulate_experiment(n_users, p_a, p_b, seed=0, share_b=0.5):
    """Simulate one A/B test.

    n_users : total number of visitors
    p_a, p_b: conversion probability in group A and B
    share_b : probability of being assigned to B (0.5 = fair coin)

    Returns a DataFrame with one row per visitor:
    user_id, variant, device, converted (0/1), revenue (0 if not converted)
    """
    rng = np.random.default_rng(seed)

    # Task 1: assign each user to "A" or "B" with probability share_b for "B".
    variant = rng.choice(["A", "B"], size=n_users, p=[1 - share_b, share_b])

    device = rng.choice(["mobile", "desktop"], size=n_users, p=[0.6, 0.4])

    # Task 2: draw the conversion: 1 with probability p_a for A users and p_b for B users, else 0.
    p = np.where(variant == "B", p_b, p_a)
    converted = (rng.random(n_users) < p).astype(int)

    # basket value of buyers: skewed, mean about 60 EUR
    basket = rng.lognormal(mean=np.log(50), sigma=0.6, size=n_users)
    revenue = converted * basket

    return pd.DataFrame({
        "user_id": np.arange(n_users),
        "variant": variant,
        "device": device,
        "converted": converted,
        "revenue": revenue.round(2),
    })

In [ ]:
# Task 3: Run the experiment with the planned number of users (2 * n_per_group),
# a true conversion rate of 5.0 % in A and 5.5 % in B. Look at the first rows.
N_USERS = int(2 * n_per_group)
df = simulate_experiment(N_USERS, p_a=0.05, p_b=0.055, seed=2026)
df.head()

In [ ]:
# Task 4: Summarize per variant: number of users, number of buyers, conversion rate.
summary = df.groupby("variant").agg(
    users=("user_id", "count"),
    buyers=("converted", "sum"),
    conversion=("converted", "mean"),
)
summary

## Exercise 3: Analyze the result 🔬

Now the real question: is the difference between A and B bigger than what noise alone would produce?

The tool: a **two-sample z-test for proportions** (`proportions_ztest`) and the **confidence interval** for the difference (`confint_proportions_2indep`).

In [ ]:
# Task 1: Run the z-test. Print the z statistic and the p-value.
buyers = summary.loc[["A", "B"], "buyers"].to_numpy()
users = summary.loc[["A", "B"], "users"].to_numpy()

z, p_value = proportions_ztest(count=buyers, nobs=users)
print(f"z = {z:.2f}, p-value = {p_value:.4f}")

In [ ]:
# Task 2: Compute the difference B - A in percentage points and its 95 % confidence interval.
diff = summary.loc["B", "conversion"] - summary.loc["A", "conversion"]
low, high = confint_proportions_2indep(buyers[1], users[1], buyers[0], users[0])
print(f"lift B - A: {diff*100:+.2f} pp, 95 % CI [{low*100:+.2f}, {high*100:+.2f}] pp")

**Task 3: Decide.** Write your decision in 2-3 sentences, as you would report it to the product team: effect, interval, p-value, sample size.

*Your answer here:*

**Task 3: Decide.** Example answer (your numbers may differ slightly):

> Variant B increased the checkout conversion rate by about +0.5 pp (95 % CI roughly +0.1 to +0.9 pp, p < 0.05) based on about 62,000 visitors. The effect is statistically significant and in line with the +0.5 pp we planned for. Recommendation: ship B, provided the guardrail metrics (load time, errors, support tickets) are unchanged.

Note how much more useful the confidence interval is than the p-value alone: the business can see that the lift could be as small as +0.1 pp.

In [ ]:
# Task 4: Did the test keep its promise?
def run_once(p_a, p_b, n_users, seed):
    d = simulate_experiment(n_users, p_a, p_b, seed=seed)
    s = d.groupby("variant")["converted"].agg(["sum", "count"])
    return proportions_ztest(count=s["sum"].to_numpy(), nobs=s["count"].to_numpy())[1]

N_SIM = 200
p_aa = [run_once(0.05, 0.05, N_USERS, seed) for seed in range(N_SIM)]
p_ab = [run_once(0.05, 0.055, N_USERS, seed) for seed in range(N_SIM)]

print(f"A/A: {np.mean(np.array(p_aa) < 0.05):.1%} of tests 'significant' (false alarms, should be ~5 %)")
print(f"A/B: {np.mean(np.array(p_ab) < 0.05):.1%} of tests 'significant' (power, should be ~80 %)")

## Exercise 4: The peeking problem 👀

In reality the data arrives day by day and everybody has a dashboard. Let's see what happens if we test **every day** and stop as soon as p < 0.05.

To make the point, we simulate an **A/A test**: there is no effect at all.

In [ ]:
def daily_pvalues(n_days, visitors_per_day, p_a, p_b, seed):
    """Return the p-value you would get if you ran the test at the end of each day (cumulative data)."""
    df = simulate_experiment(n_days * visitors_per_day, p_a, p_b, seed=seed)
    df["day"] = np.repeat(np.arange(1, n_days + 1), visitors_per_day)
    pvals = []
    for day in range(1, n_days + 1):
        # Task 1: take all data up to and including `day`, compute buyers/users per variant and the p-value.
        so_far = df[df["day"] <= day].groupby("variant")["converted"].agg(["sum", "count"])
        _, p = proportions_ztest(count=so_far["sum"].to_numpy(), nobs=so_far["count"].to_numpy())
        pvals.append(p)
    return np.array(pvals)

In [ ]:
# Task 2: Plot the daily p-value of 5 A/A tests (30 days, 2000 visitors/day, p_a = p_b = 0.05) in one chart,
# with a horizontal line at 0.05. How many of them dip below the line at some point?
plt.figure(figsize=(8, 4))
for seed in range(5):
    pv = daily_pvalues(30, VISITORS_PER_DAY, 0.05, 0.05, seed=seed)
    plt.plot(range(1, 31), pv, label=f"A/A test #{seed}" + ("  (dips below!)" if pv.min() < 0.05 else ""))
plt.axhline(0.05, ls="--", color="red")
plt.xlabel("day")
plt.ylabel("p-value if you test today")
plt.legend()
plt.show()

In [ ]:
# Task 3: Repeat for 100 A/A tests.
N_SIM = 100
all_pvals = np.array([daily_pvalues(30, VISITORS_PER_DAY, 0.05, 0.05, seed=s) for s in range(N_SIM)])

at_end = (all_pvals[:, -1] < 0.05).mean()
ever = (all_pvals.min(axis=1) < 0.05).mean()
print(f"significant at day 30:        {at_end:.0%}  (the promised false alarm rate)")
print(f"significant on at least 1 day: {ever:.0%}  (what a daily peeker gets)")

**Task 4: Explain** in your own words why peeking breaks the test, and what you should do instead.

*Your answer here:*

**Task 4: Explain.** Example answer:

> The p-value is only valid for one look at a fixed, pre-planned sample size. If you look every day, you give noise 30 chances to cross the 5 % line instead of one, so the real false alarm rate is several times higher. What to do: compute the sample size upfront, run until you reach it, test once. Looking at the dashboard is fine, deciding on it is not. If you really need to stop early, use a method built for it (sequential testing).

## Exercise 5: A continuous metric: revenue per visitor 💶

Conversion is binary (0/1). Revenue per visitor is continuous: most visitors spend 0, buyers spend a skewed amount.
For means we use the **t-test** (`scipy.stats.ttest_ind`).

In [ ]:
# Task 1: Using `df` from Exercise 2: plot a histogram of the revenue of BUYERS only (revenue > 0), per variant.
buyers_df = df[df["revenue"] > 0]
plt.figure(figsize=(8, 4))
for v in ["A", "B"]:
    plt.hist(buyers_df.loc[buyers_df["variant"] == v, "revenue"], bins=60, alpha=0.6, label=v)
plt.xlabel("revenue of a buyer (EUR)")
plt.ylabel("number of buyers")
plt.legend()
plt.show()
# Strongly right-skewed: many small baskets, a long tail of big ones. The mean is pulled up by the tail.

In [ ]:
# Task 2: Compute the mean revenue PER VISITOR (including the zeros) for A and B,
# then run a t-test (Welch: equal_var=False) and print the p-value and the 95 % confidence interval of the difference.
rev_a = df.loc[df["variant"] == "A", "revenue"]
rev_b = df.loc[df["variant"] == "B", "revenue"]
print(f"revenue per visitor: A = {rev_a.mean():.3f} EUR, B = {rev_b.mean():.3f} EUR")

res = stats.ttest_ind(rev_b, rev_a, equal_var=False)
ci = res.confidence_interval()
print(f"difference B - A = {rev_b.mean() - rev_a.mean():+.3f} EUR, 95 % CI [{ci.low:+.3f}, {ci.high:+.3f}], p = {res.pvalue:.4f}")

**Task 3:** Compare with the conversion result from Exercise 3. Is the revenue result as clear? Why (not)?

*Your answer here:*

**Task 3:** Example answer:

> The conversion lift was clearly significant, the revenue lift usually is not (or only barely), even though B converts more and buyers spend the same in both groups. Revenue per visitor is far noisier: 95 % zeros plus a long-tailed basket value. A noisier metric needs many more users for the same effect. That is why conversion is the primary metric and revenue a secondary one here.

## Exercise 6: Sanity checks 🩺

Before anyone believes an experiment result, two checks:

1. **Sample ratio mismatch (SRM):** we planned a 50/50 split. Is the observed split compatible with that?
2. **Segments:** does the result hold in every segment, or is something odd going on (Simpson's paradox)?

In [ ]:
# Task 1: SRM check for `df` from Exercise 2.
counts = df["variant"].value_counts().sort_index()
print(counts.to_dict(), "p =", round(stats.chisquare(counts).pvalue, 4), "-> fine" )

df_broken = simulate_experiment(100_000, 0.05, 0.055, seed=7, share_b=0.49)
counts_broken = df_broken["variant"].value_counts().sort_index()
print(counts_broken.to_dict(), "p =", round(stats.chisquare(counts_broken).pvalue, 6), "-> SRM! Do not analyze, find the bug first")
# 49/51 looks harmless, but with 100,000 users it is extremely unlikely to happen by chance.
# Typical causes: a redirect that fails on some browsers, bots filtered in one variant only, caching.

In [ ]:
# Task 2: Segment check.
pivot = df.pivot_table(index="device", columns="variant", values="converted", aggfunc="mean")
pivot["lift_pp"] = (pivot["B"] - pivot["A"]) * 100
pivot
# With a fair coin flip the device mix is the same in A and B, so B should be ahead on both devices
# (up to noise: the segments are smaller, so the lifts are noisier than the overall one).

In [ ]:
# Task 3 (Simpson's paradox): Build the case from the lecture by hand and check it.
simpson = pd.DataFrame({
    "device": ["mobile", "mobile", "desktop", "desktop"],
    "variant": ["A", "B", "A", "B"],
    "users": [1000, 4000, 4000, 1000],
    "conversion": [0.03, 0.035, 0.08, 0.09],
})
simpson["buyers"] = simpson["users"] * simpson["conversion"]
overall = simpson.groupby("variant")[["users", "buyers"]].sum()
overall["conversion"] = overall["buyers"] / overall["users"]
display(simpson.pivot(index="device", columns="variant", values="conversion"))
display(overall)
# B wins in both segments but loses overall, because B got mostly mobile users (low conversion)
# and A mostly desktop users. The randomization was not a fair coin per user: the device mix differs.
# That is a broken experiment, not a real effect. The segment check reveals it.

## Optional: a real dataset 🌍

If you want to try this on real data, the Kaggle dataset
[Marketing A/B Testing](https://www.kaggle.com/datasets/faviovaz/marketing-ab-testing) contains about 588,000 users
who saw either an ad (`test group = "ad"`) or a public service announcement (`"psa"`), and whether they converted.

1. Download `marketing_AB.csv` from Kaggle (free account) and upload it to Colab (folder icon on the left).
2. Load it with `pd.read_csv("marketing_AB.csv")`.
3. Compute users, buyers and the conversion rate per `test group`.
4. Run the SRM check. (Spoiler: the split is far from 50/50. Is that a bug, or was it planned that way?)
5. Run the z-test and compute the confidence interval of the lift.
6. Is the lift relevant for the business? What else would you want to know before recommending the ad?

## Summary

- **Plan first:** baseline, MDE, α and power give you the sample size. Without it you do not know when to stop.
- **Randomize** per user with a fair coin. Check the split (SRM).
- **One test, at the end.** Peeking turns 5 % false alarms into 25 % or more.
- **Report effect + confidence interval**, not just "significant".
- **Noisy metrics need more users.** Conversion before revenue.
- **Check segments.** A result that only holds overall is suspicious.